In [12]:
import pandas as pd
from calendar import monthrange
from sqlalchemy import create_engine, Date, Integer, String
import datetime

# ==============================
# 1. CONEXÃO COM BANCO
# ==============================
engine = create_engine("sqlite:///DBVendas.db")

# ==============================================================
# FERIADOS NACIONAIS DO BRASIL
# ==============================================================
def pascoa(ano):
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = ((h + l - 7 * m + 114) % 31) + 1
    return pd.Timestamp(ano, mes, dia)

def gerar_feriados_nacionais(anos):
    feriados = set()
    fixos = [(1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)]
    for ano in anos:
        for mes, dia in fixos:
            feriados.add(pd.Timestamp(ano, mes, dia))
        if ano >= 2024:
            feriados.add(pd.Timestamp(ano, 11, 20))
        p = pascoa(ano)
        feriados.add(p - pd.Timedelta(days=2)) # Sexta-Feira Santa
        feriados.add(p + pd.Timedelta(days=60)) # Corpus Christi
    return feriados

# ==============================================================
# CONFIGURAÇÕES
# ==============================================================
DATA_INICIO = "2010-01-01"
DATA_FIM    = "2026-12-31"

dias_semana    = {"Mon": "Seg", "Tue": "Ter", "Wed": "Qua", "Thu": "Qui", "Fri": "Sex", "Sat": "Sáb", "Sun": "Dom"}
meses_abrev    = {"Jan": "Jan", "Feb": "Fev", "Mar": "Mar", "Apr": "Abr", "May": "Mai", "Jun": "Jun", "Jul": "Jul", "Aug": "Ago", "Sep": "Set", "Oct": "Out", "Nov": "Nov", "Dec": "Dez"}
meses_completo = {"January": "Janeiro", "February": "Fevereiro", "March": "Março", "April": "Abril", "May": "Maio", "June": "Junho", "July": "Julho", "August": "Agosto", "September": "Setembro", "October": "Outubro", "November": "Novembro", "December": "Dezembro"}

# 1. Gerar Range e Feriados
data_range = pd.date_range(start=DATA_INICIO, end=DATA_FIM, freq="D")
feriados_br = gerar_feriados_nacionais(range(data_range.year.min(), data_range.year.max() + 1))

# 2. Criar DataFrame Base
df = pd.DataFrame({"Data_Full": data_range})

# 3. Cálculos de Flags e Datas
e_feriado = df["Data_Full"].isin(feriados_br)
e_fim_semana = df["Data_Full"].dt.weekday >= 5
df["Dia_Util_Bool"] = ~e_fim_semana & ~e_feriado

# --- Colunas de Datas (Formato DATE para SQL/Excel) ---
df["Data"] = df["Data_Full"].dt.date
df["Inicio_da_Semana"] = (df["Data_Full"] - pd.to_timedelta(df["Data_Full"].dt.weekday, unit="D")).dt.date
df["Fim_da_Semana"] = (pd.to_datetime(df["Inicio_da_Semana"]) + pd.Timedelta(days=6)).dt.date
df["Inicio_do_Mes"] = df["Data_Full"].dt.to_period("M").dt.to_timestamp().dt.date
df["Fim_do_Mes"] = df["Data_Full"].apply(lambda x: datetime.date(x.year, x.month, monthrange(x.year, x.month)[1]))

# --- Contadores de Dias Úteis ---
#df["ano_mes"] = df["Data_Full"].dt.to_period("M")
#df["ano"] = df["Data_Full"].dt.year.values
#df["ano"] = df["ano"].astype(int)

#aux_Dia_Util_do_Mes = df.groupby("ano_mes")["Dia_Util_Bool"].cumsum().astype(int)
#df["Dia_Util_do_Mes"] = aux_Dia_Util_do_Mes.astype(str)

#df["Dia_Util_do_Mes"] = df.groupby("ano_mes")["Dia_Util_Bool"].cumsum().astype(int)
#df["Dia_Util_do_Mes"] = df["Dia_Util_do_Mes"].fillna(0).astype(int)
#df["Dia_Util_do_Ano"] = df.groupby("ano")["Dia_Util_Bool"].cumsum().astype(int)
#df["Dia_Util_do_Ano"] = df["Dia_Util_do_Ano"].fillna(0).astype(int)
# Total de dias úteis no mês
#uteis_no_mes_map = df.groupby("ano_mes")["Dia_Util_Bool"].sum().astype(int)
#df["Dias_Uteis_no_Mes"] = df["ano_mes"].map(uteis_no_mes_map).astype(int)
#df["Dias_Uteis_no_Mes"] = df["Dias_Uteis_no_Mes"].fillna(0).astype(int)

# Zerar contadores em dias não úteis
#df.loc[~df["Dia_Util_Bool"], ["Dia_Util_do_Mes", "Dia_Util_do_Ano"]] = 0

# --- Contadores de Dias Úteis ---
df["ano_mes"] = df["Data_Full"].dt.to_period("M")
df["ano_mes_str"] = df["Data_Full"].dt.strftime("%Y-%m")   # ← chave string para groupby
df["ano"] = df["Data_Full"].dt.year.astype(int)

df["Dia_Util_do_Mes"] = (
    df.groupby("ano_mes_str")["Dia_Util_Bool"]
    .cumsum()
    .astype("int64")
)

df["Dia_Util_do_Ano"] = (
    df.groupby("ano")["Dia_Util_Bool"]
    .cumsum()
    .astype("int64")
)

# Total de dias úteis no mês
uteis_no_mes_map = (
    df.groupby("ano_mes_str")["Dia_Util_Bool"]
    .transform("sum")
    .astype("int64")
)
df["Dias_Uteis_no_Mes"] = uteis_no_mes_map

# Zerar contadores em dias não úteis
df.loc[~df["Dia_Util_Bool"], ["Dia_Util_do_Mes", "Dia_Util_do_Ano"]] = 0

# --- Colunas de Texto e Formatação ---
df["Dia"] = df["Data_Full"].dt.day.astype(int)
df["Num_Dia_semana"] = (df["Data_Full"].dt.weekday + 1).astype(int)
df["Dia_da_Semana"] = df["Data_Full"].dt.strftime("%a").map(dias_semana)
df["Mes"] = df["Data_Full"].dt.month.astype(int)
df["Mes_Ano"] = df["Data_Full"].dt.strftime("%m/%Y")
df["Ini_Mes_Ano"] = df["Data_Full"].dt.strftime("%b/%Y").map(lambda x: f"{meses_abrev.get(x[:3], x[:3])}/{x[4:]}")
df["Inicial_do_Mes"] = df["Data_Full"].dt.strftime("%b").map(meses_abrev)
df["Nome_do_Mes"] = df["Data_Full"].dt.strftime("%B").map(meses_completo)
df["Trimestre"] = df["Data_Full"].dt.quarter.astype(int)
df["Trimestre_Ano"] = df.apply(lambda x: f"Tri {x['Data_Full'].quarter}-{x['Data_Full'].year}", axis=1)
df["Ano_Txt"] = df["Data_Full"].dt.year.astype(str)
df["Semana_do_Ano"] = df["Data_Full"].dt.isocalendar().week.astype(int)
df["Dia_Util"] = df["Dia_Util_Bool"].map({True: "Sim", False: "Não"})
df["Feriado_Nacional"] = e_feriado.map({True: "Sim", False: "Não"})
df["Semana_Ano"] = df.apply(lambda x: f"Sem {x['Data_Full'].isocalendar().week}-{x['Data_Full'].isocalendar().year}", axis=1)
df["Dias_no_Mes"] = df["Data_Full"].apply(lambda x: monthrange(x.year, x.month)[1]).astype(int)
df["E_1_Dia_do_Mes"] = df["Data_Full"].dt.day.apply(lambda x: "Sim" if x == 1 else "Não")
df["E_Ultimo_Dia_do_Mes"] = (df["Data"] == df["Fim_do_Mes"]).map({True: "Sim", False: "Não"})
df["Dia_Nao_Util"] = (~df["Dia_Util_Bool"]).map({True: "Sim", False: "Não"})

# Seleção final de colunas e ordem
colunas_finais = [
    "Data", "Dia", "Num_Dia_semana", "Dia_da_Semana", "Mes", "Mes_Ano", 
    "Ini_Mes_Ano", "Inicial_do_Mes", "Nome_do_Mes", "Trimestre", "Trimestre_Ano", 
    "Ano_Txt", "Semana_do_Ano", "Dia_Util", "Feriado_Nacional", "Inicio_da_Semana", 
    "Fim_da_Semana", "Semana_Ano", "Inicio_do_Mes", "Fim_do_Mes", "Dias_no_Mes", 
    "Dia_Util_do_Mes", "Dia_Util_do_Ano", "Dias_Uteis_no_Mes", "E_1_Dia_do_Mes", 
    "E_Ultimo_Dia_do_Mes", "Dia_Nao_Util"
]

df_final = df[colunas_finais].rename(columns={"Ano_Txt": "Ano"})

# ==============================================================
# SALVAR
# ==============================================================

# Mapeamento para garantir o tipo DATE no Banco de Dados
colunas_data = ["Data", "Inicio_da_Semana", "Fim_da_Semana", "Inicio_do_Mes", "Fim_do_Mes"]
dtype_map = {col: Date for col in colunas_data}

# Forçar explicitamente como INTEGER no schema da tabela
dtype_map["Dia_Util_do_Mes"]    = Integer
dtype_map["Dia_Util_do_Ano"]    = Integer
dtype_map["Dias_Uteis_no_Mes"]  = Integer

# Salvar no SQLite
df_final.to_sql("calendario", engine, if_exists="replace", index=False, dtype=dtype_map)

# Salvar no Excel
df_final.to_excel("Calendario.xlsx", index=False, engine="openpyxl")

print("Calendário gerado com sucesso!")
print(f"Total de linhas: {len(df_final)}")

Calendário gerado com sucesso!
Total de linhas: 6209
